In [2]:
!pip install transformers torch --index-url https://download.pytorch.org/whl/cpu
!pip install accelerate

Looking in indexes: https://download.pytorch.org/whl/cpu
ERROR: Could not find a version that satisfies the requirement transformers (from versions: none)
ERROR: No matching distribution found for transformers
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 674.2 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 612.9/612.9 kB 850.5 kB/s eta 0:00:0000:010:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.2/507.2 kB 661.8 kB/s eta 0:00:0000:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 558.2 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 978.3 kB/s eta 0:00:000:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 560.3 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 408.9 kB/s eta 0:00:00--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.3/108.3 kB 153.2 kB/s eta 0:00:00a 0:00:01
  Attempting uninstall: click
    Found existing installation: click 8.1.7
   

In [3]:
import sys
# --extra-index-url tells pip to check the standard store AND the CPU-only store
!{sys.executable} -m pip install transformers
!{sys.executable} -m pip install torch --index-url https://download.pytorch.org/whl/cpu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 86.5 kB/s eta 0:00:0000:01m00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 800.2/800.2 kB 83.4 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 245.4 kB/s eta 0:00:0000:0100:01
Looking in indexes: https://download.pytorch.org/whl/cpu


In [5]:
import os
import json
import numpy as np
from transformers import pipeline

# --- Configuration (Fixed for your environment) ---
MODEL_NAME = 'gpt2'
# Using relative path to stay within your /work/ directory permissions
OUTPUT_DIR = './generated_data' 
NUM_LOGS_TO_GENERATE = 10

# Ensure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Environment check: Paths configured. Memory-safe model selected.")

try:
    # 1. Load the text generation pipeline
    print(f"Loading model: {MODEL_NAME} into RAM (approx 500MB)...")
    # Setting device=-1 forces CPU usage, saving your VM's video memory
    generator = pipeline('text-generation', model=MODEL_NAME, device=-1)
    print("Model loaded successfully.")

    # 2. Define the security-specific prompt
    prompt = (
        "Generate a single, valid JSON object representing a firewall log entry. "
        "The log should indicate a blocked inbound connection attempt from a suspicious IP. "
        "Fields: \"timestamp\", \"source_ip\", \"destination_ip\", \"destination_port\", "
        "\"protocol\", \"action\": \"DENY\", \"reason\": \"Suspicious IP\". "
        "JSON output only:"
    )

    generated_logs = []
    print(f"Generating {NUM_LOGS_TO_GENERATE} synthetic logs. Please wait...")

    for i in range(NUM_LOGS_TO_GENERATE):
        # Generate text
        outputs = generator(
            prompt,
            max_length=200,
            num_return_sequences=1,
            temperature=0.8,
            pad_token_id=generator.tokenizer.eos_token_id,
            truncation=True
        )

        full_text = outputs[0]['generated_text']

        # 3. Extract JSON from the 'chatty' LLM output
        start = full_text.find('{')
        end = full_text.rfind('}')

        if start != -1 and end != -1 and start < end:
            json_str = full_text[start:end+1]
            try:
                log_data = json.loads(json_str)
                
                # Basic validation: ensure it's not just an empty dict
                if "source_ip" in log_data:
                    generated_logs.append(log_data)
                    print(f" [+] Log {i+1} parsed successfully.")
                else:
                    print(f" [-] Log {i+1} missing fields. Retrying...")
            except json.JSONDecodeError:
                print(f" [!] Log {i+1} had malformed JSON. Skipping.")
        else:
            print(f" [!] Log {i+1} no JSON found in output.")

    # 4. Save to JSON Lines (.jsonl) format
    output_file = os.path.join(OUTPUT_DIR, 'synthetic_firewall_logs.jsonl')
    with open(output_file, 'w') as f:
        for entry in generated_logs:
            f.write(json.dumps(entry) + '\n')

    print(f"\n--- SUCCESS ---")
    print(f"Saved {len(generated_logs)} logs to: {output_file}")

except Exception as e:
    print(f"\n[ERROR] An unexpected issue occurred: {e}")
    print("Tip: If the kernel died, restart it and ensure other notebooks are closed.")

print("\nLab Ready for Evaluation.")

Environment check: Paths configured. Memory-safe model selected.
Loading model: gpt2 into RAM (approx 500MB)...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Model loaded successfully.
Generating 10 synthetic logs. Please wait...


Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 [!] Log 1 had malformed JSON. Skipping.


KeyboardInterrupt: 

In [ ]:
#original : 

In [ ]:
import os
import json
from transformers import pipeline

# --- Configuration ---
# Using a smaller model for faster execution in a typical laptop environment.
# For higher fidelity, consider 'gpt2-medium' or 'gpt2-large' if resources allow.
MODEL_NAME = 'gpt2'
OUTPUT_DIR = '/home/student/cyberai-lab/generated_data'
NUM_LOGS_TO_GENERATE = 10

# Ensure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"If you see this message → your lab is working perfectly!")

try:
    # Load the text generation pipeline
    print(f"Loading model: {MODEL_NAME}...")
    generator = pipeline('text-generation', model=MODEL_NAME)
    print("Model loaded successfully.")

    # Define a detailed prompt for generating JSON firewall logs
    prompt = "Generate a single, valid JSON object representing a firewall log entry. The log should indicate a blocked inbound connection attempt from a suspicious IP address to a web server port. Include the following fields: \"timestamp\" (ISO 8601 format), \"source_ip\", \"destination_ip\", \"destination_port\", \"protocol\", \"action\" (should be \"DENY\"), and \"reason\" (e.g., \"Suspicious IP\"). Ensure the JSON is correctly formatted."

    generated_logs = []
    print(f"Generating {NUM_LOGS_TO_GENERATE} synthetic log entries...")

    for i in range(NUM_LOGS_TO_GENERATE):
        # Generate text based on the prompt
        # Adjust max_length to ensure enough space for a full JSON object
        # Temperature controls randomness: lower = more predictable, higher = more creative
        outputs = generator(
            prompt,
            max_length=250,  # Increased length to accommodate JSON structure
            num_return_sequences=1,
            temperature=0.8,
            pad_token_id=generator.tokenizer.eos_token_id # Important for preventing warnings
        )

        generated_text = outputs[0]['generated_text']

        # The LLM might repeat the prompt or add extra text. We need to extract the JSON.
        # Find the start and end of the JSON object
        json_start = generated_text.find('{')
        json_end = generated_text.rfind('}')

        if json_start != -1 and json_end != -1 and json_start < json_end:
            json_string = generated_text[json_start : json_end + 1]
            try:
                log_entry = json.loads(json_string)
                # Basic validation: check if essential fields are present
                required_fields = ["timestamp", "source_ip", "destination_ip", "destination_port", "protocol", "action", "reason"]
                if all(field in log_entry for field in required_fields):
                    generated_logs.append(log_entry)
                    print(f"Successfully generated and parsed log {i+1}/{NUM_LOGS_TO_GENERATE}")
                else:
                    print(f"Warning: Log {i+1} missing required fields. Skipping.")
            except json.JSONDecodeError:
                print(f"Warning: Failed to parse JSON for log {i+1}. Raw output snippet: {json_string[:100]}...")
        else:
            print(f"Warning: Could not find valid JSON structure in output for log {i+1}.")

    # Save the generated logs to a file
    output_filename = os.path.join(OUTPUT_DIR, 'synthetic_firewall_logs.jsonl')
    with open(output_filename, 'w') as f:
        for log in generated_logs:
            f.write(json.dumps(log) + '\n')

    print(f"\nSuccessfully generated and saved {len(generated_logs)} synthetic log entries to {output_filename}")
    print("Expected output: You should see a list of generated JSON log entries, with minimal warnings about parsing errors.")

except ImportError:
    print("Error: The 'transformers' or 'torch' library is not installed. Please run 'pip install transformers torch' or ensure your environment is set up correctly.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

print("\n--- Lab Reset Command ---")
print("To reset the lab environment, run: docker compose down && docker compose up -d")

In [ ]:
import os
import json
from transformers import pipeline

# --- Configuration ---
MODEL_NAME = 'gpt2'
OUTPUT_DIR = '/home/student/cyberai-lab/generated_data'
NUM_LOGS_TO_GENERATE = 10

# Ensure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"If you see this message → your lab is working perfectly!")

try:
    # Load the text generation pipeline
    print(f"Loading model: {MODEL_NAME}...")
    generator = pipeline('text-generation', model=MODEL_NAME)
    print("Model loaded successfully.")

    # Define a detailed prompt for generating JSON firewall logs
    prompt = "Generate a single, valid JSON object representing a firewall log entry. The log should indicate a blocked inbound connection attempt from a suspicious IP address to a web server port. Include the following fields: \"timestamp\" (ISO 8601 format), \"source_ip\", \"destination_ip\", \"destination_port\", \"protocol\", \"action\" (should be \"DENY\"), and \"reason\" (e.g., \"Suspicious IP\"). Ensure the JSON is correctly formatted."

    generated_logs = []
    print(f"Generating {NUM_LOGS_TO_GENERATE} synthetic log entries...")

    for i in range(NUM_LOGS_TO_GENERATE):
        outputs = generator(
            prompt,
            max_length=250,
            num_return_sequences=1,
            temperature=0.8,
            pad_token_id=generator.tokenizer.eos_token_id
        )

        generated_text = outputs[0]['generated_text']

        json_start = generated_text.find('{')
        json_end = generated_text.rfind('}')

        if json_start != -1 and json_end != -1 and json_start < json_end:
            json_string = generated_text[json_start : json_end + 1]
            try:
                log_entry = json.loads(json_string)
                required_fields = ["timestamp", "source_ip", "destination_ip", "destination_port", "protocol", "action", "reason"]
                if all(field in log_entry for field in required_fields):
                    generated_logs.append(log_entry)
                    print(f"Successfully generated and parsed log {i+1}/{NUM_LOGS_TO_GENERATE}")
                else:
                    print(f"Warning: Log {i+1} missing required fields. Skipping.")
            except json.JSONDecodeError:
                print(f"Warning: Failed to parse JSON for log {i+1}. Raw output snippet: {json_string[:100]}...")
        else:
            print(f"Warning: Could not find valid JSON structure in output for log {i+1}.")

    output_filename = os.path.join(OUTPUT_DIR, 'synthetic_firewall_logs.jsonl')
    with open(output_filename, 'w') as f:
        for log in generated_logs:
            f.write(json.dumps(log) + '\n')

    print(f"\nSuccessfully generated and saved {len(generated_logs)} synthetic log entries to {output_filename}")
    print("Expected output: You should see a list of generated JSON log entries, with minimal warnings about parsing errors.")

except ImportError:
    print("Error: The 'transformers' or 'torch' library is not installed. Please run 'pip install transformers torch' or ensure your environment is set up correctly.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

print("\n--- Lab Reset Command ---")
print("To reset the lab environment, run: docker compose down && docker compose up -d")